# Sprint 4 — Turing College Project

Welcome!
In this project, we explore the Coursera Courses Dataset from Kaggle and use it as a playground to practice and demonstrate data analysis and visualization skills.

### Notebook outline
1. Importing Data  
2. Data Cleaning & Preparation   
3. Exploratory Data Analysis & Visualization   


## Importing Data

In [1]:
import opendatasets as od
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re
from collections import Counter

In [2]:
dataset = 'https://www.kaggle.com/datasets/siddharthm1698/coursera-course-dataset'

In [3]:
def data_from_kaggle(dataset):
    """
    Download a Kaggle dataset using opendatasets and return the loaded DataFrame. Works with only one csv
    
    Args:
        dataset: Kaggle dataset URL.
    
    Returns:
        df (pd.DataFrame): CSV file found in the dataset folder, converted to pd.DataFrame
    """
    data_dir =  od.download(dataset)
    if data_dir is None:
        # Extract folder name from URL
        folder_name = dataset.split('/')[-1]
        data_dir = f'./{folder_name}'
    file = os.listdir(data_dir)
    csv_path = os.path.join(data_dir, file[0])
    df = pd.read_csv(csv_path)
    return df
    

In [4]:
cours = data_from_kaggle(dataset) #you'll need your Kaggle credentials (username and API key) to use that

Skipping, found downloaded files in ".\coursera-course-dataset" (use force=True to force download)


In [5]:
cours.head()

,Unnamed: 0,course_title,course_organization,course_Certificate_type,course_rating,course_difficulty,course_students_enrolled
0,134,(ISC)² Systems Security Certified Practitioner...,(ISC)²,SPECIALIZATION,4.7,Beginner,5.3k
1,743,A Crash Course in Causality: Inferring Causal...,University of Pennsylvania,COURSE,4.7,Intermediate,17k
2,874,A Crash Course in Data Science,Johns Hopkins University,COURSE,4.5,Mixed,130k
3,413,A Law Student's Toolkit,Yale University,COURSE,4.7,Mixed,91k
4,635,A Life of Happiness and Fulfillment,Indian School of Business,COURSE,4.8,Mixed,320k


In [6]:
cours.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 7 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Unnamed: 0                891 non-null    int64  
 1   course_title              891 non-null    object 
 2   course_organization       891 non-null    object 
 3   course_Certificate_type   891 non-null    object 
 4   course_rating             891 non-null    float64
 5   course_difficulty         891 non-null    object 
 6   course_students_enrolled  891 non-null    object 
dtypes: float64(1), int64(1), object(5)
memory usage: 48.9+ KB


We can see that the dataset contains no null values and that the column names are generally clear, although overly long. There is also an “Unnamed” column that carries no useful information, and the course_students_enrolled column is stored as an object type even though it should logically be numeric. Let’s address these issues and clean the dataset.

Defining additional function

In [7]:
def m_to_k(x):
    x = str(x)

    if x.endswith('m'):              
        return float(x[:-1]) * 1000 
    else:
        return float(x) 

## Data Cleaning & Preparation

In [8]:
cours_cleaned = cours.drop(columns = ["Unnamed: 0"]) #dropping index column

In [9]:
cours_cleaned = cours_cleaned.drop_duplicates() #dropping duplicates if any
cours_cleaned.shape 

(891, 6)

Number of entries is still the same so there were no duplicates

Just to be 100% sure let's check that there are no null values at this dataset

In [10]:
cours_cleaned.isnull().sum()

course_title                0
course_organization         0
course_Certificate_type     0
course_rating               0
course_difficulty           0
course_students_enrolled    0
dtype: int64

In [11]:
#renaming columns
cours_cleaned = cours_cleaned.rename( columns = {
    'course_title':'Title',
    'course_organization': 'Organization',
    'course_Certificate_type': 'Certificate', 
    'course_rating' : 'Rating', 
    'course_difficulty' : 'Difficulty', 
    'course_students_enrolled': 'Students(thousands)' 
}
                                    )

In [12]:
cours_cleaned.head()

,Title,Organization,Certificate,Rating,Difficulty,Students(thousands)
0,(ISC)² Systems Security Certified Practitioner...,(ISC)²,SPECIALIZATION,4.7,Beginner,5.3k
1,A Crash Course in Causality: Inferring Causal...,University of Pennsylvania,COURSE,4.7,Intermediate,17k
2,A Crash Course in Data Science,Johns Hopkins University,COURSE,4.5,Mixed,130k
3,A Law Student's Toolkit,Yale University,COURSE,4.7,Mixed,91k
4,A Life of Happiness and Fulfillment,Indian School of Business,COURSE,4.8,Mixed,320k


Let's make Students column numerical. Immediately we can see that the easiest way to do that is to get rid of k at the end

In [13]:
cours_cleaned['Students(thousands)'] = cours_cleaned['Students(thousands)'].str.replace('k', '').str.strip()

In [14]:
cours_cleaned['Students(thousands)'].unique()  #checking if there are any other symbols that prevent us from changing type

array(['5.3', '17', '130', '91', '320', '39', '350', '2.4', '61', '12',
       '4', '13', '11', '27', '110', '6.6', '540', '22', '2.9', '80',
       '9.9', '23', '9.2', '78', '190', '35', '29', '14', '21', '94',
       '69', '40', '220', '150', '18', '270', '7.9', '30', '36', '20',
       '8.1', '120', '71', '63', '42', '480', '97', '200', '180', '4.2',
       '310', '3.9', '79', '31', '15', '10', '66', '33', '56', '7.3',
       '9.7', '210', '28', '6.5', '55', '2.3', '8.8', '88', '1.9', '68',
       '1.6', '9.5', '57', '26', '84', '95', '5.8', '24', '67', '280',
       '38', '77', '510', '89', '48', '160', '32', '340', '82', '790',
       '19', '51', '4.8', '37', '43', '6.4', '49', '240', '46', '7.8',
       '75', '81', '140', '5.5', '99', '100', '3', '830', '740', '60',
       '96', '690', '44', '4.5', '8.2', '16', '300', '8', '41', '54', '9',
       '380', '58', '5.6', '7.1', '83', '3.4', '1.5', '230', '760', '86',
       '45', '7.2', '1.8', '4.1', '76', '490', '170', '260', '65', '

We can also see that there are some millions as well, so we have to change that to thousands and then delete m.  
Let's look closer at those entries

In [15]:
cours_cleaned[cours_cleaned['Students(thousands)'].str.contains('m', case=False)]

,Title,Organization,Certificate,Rating,Difficulty,Students(thousands)
564,Machine Learning,Stanford University,COURSE,4.9,Mixed,3.2m
674,Programming for Everybody (Getting Started wit...,University of Michigan,COURSE,4.8,Mixed,1.3m
688,Python for Everybody,University of Michigan,SPECIALIZATION,4.8,Beginner,1.5m
815,The Science of Well-Being,Yale University,COURSE,4.9,Mixed,2.5m


There are only 4 courses and each of them has a decimal value.  
Since each million contains 1000 thousands, and since, for example 3.2m = 3200 thousands, what we'll do is just replace m with 00 and delete a period

In [16]:
cours_cleaned['Students(thousands)'] = cours_cleaned['Students(thousands)'].apply(m_to_k)

In [17]:
np.set_printoptions(suppress=True) #suppressing scientific notation 

In [18]:
#sanity check let's see if only right values were affected
cours_cleaned['Students(thousands)'] .unique()

array([   5.3,   17. ,  130. ,   91. ,  320. ,   39. ,  350. ,    2.4,
         61. ,   12. ,    4. ,   13. ,   11. ,   27. ,  110. ,    6.6,
        540. ,   22. ,    2.9,   80. ,    9.9,   23. ,    9.2,   78. ,
        190. ,   35. ,   29. ,   14. ,   21. ,   94. ,   69. ,   40. ,
        220. ,  150. ,   18. ,  270. ,    7.9,   30. ,   36. ,   20. ,
          8.1,  120. ,   71. ,   63. ,   42. ,  480. ,   97. ,  200. ,
        180. ,    4.2,  310. ,    3.9,   79. ,   31. ,   15. ,   10. ,
         66. ,   33. ,   56. ,    7.3,    9.7,  210. ,   28. ,    6.5,
         55. ,    2.3,    8.8,   88. ,    1.9,   68. ,    1.6,    9.5,
         57. ,   26. ,   84. ,   95. ,    5.8,   24. ,   67. ,  280. ,
         38. ,   77. ,  510. ,   89. ,   48. ,  160. ,   32. ,  340. ,
         82. ,  790. ,   19. ,   51. ,    4.8,   37. ,   43. ,    6.4,
         49. ,  240. ,   46. ,    7.8,   75. ,   81. ,  140. ,    5.5,
         99. ,  100. ,    3. ,  830. ,  740. ,   60. ,   96. ,  690. ,
      

Let's also ensure logical order of difficulty categories, so that plotting would be easier to interpret

In [19]:
#ensure logical order
cours_cleaned["Difficulty"] = pd.Categorical(
    cours_cleaned["Difficulty"],
    categories=["Beginner", "Intermediate", "Advanced", "Mixed"],
    ordered=True
)

NameError: name 'df' is not defined

Everything worked correct!

In [ ]:
cours_cleaned.info()

We’ve completed the data preparation stage and now have everything needed for basic EDA and visualizations.

#### Possible Improvements

For columns like Certificate and Difficulty, we could have applied one-hot encoding to convert the categorical values into numerical features. This would be useful if we planned to build machine learning models or compute correlations involving these variables. However, since this project focuses solely on exploratory analysis and visualizations, one-hot encoding is not necessary, and we keep the original categorical format.

## Exploratory Data Analysis & Visualization

In this part we'll answer following questions:

**Course Exploration**

1. How many courses are in the dataset?  

2. Which courses have the highest number of enrollments?  

3. What are the most common words in course titles?  

**Certification Exploration**

4. What certification types are present in the dataset?

5. How many courses are offered per certification type?

6. How many students are enrolled per certification type?

7. What is the relationship between course difficulty and certification type?

**Organization Exploration**

8. How many distinct organizations are in the dataset?

9. Which are the top 10 organizations by number of courses?

10. Which organizations have the highest average course rating?

11. What are the average course ratings for the top 10 organizations by number of courses?

12. How many students are enrolled in courses offered by the top 10 organizations?

13. How do the top 10 organizations compare across key metrics (number of courses, ratings, enrollments)?

**Rating Exploration**

14. What is the distribution of course ratings?

15. What is the relationship between course ratings and student enrollment?

16. What is the relationship between course ratings and course difficulty?

17. What is the relationship between course ratings and certification type?

In [ ]:
df = cours_cleaned.copy() #copy our cleaned dataset

**Course exploration**

1. How many courses are in the dataset?

In [ ]:
print(f"There are {len(df)} courses in this dataset")

2. Which courses have the highest number of enrollments?


In [ ]:
top_cours = df.sort_values(by="Students(thousands)", ascending=False)
top_cours = top_cours[:10]

print(f'Top courses by students enrolled (in thousands):')
for i in range(10):
    print(f'{i+1}. {top_cours["Title"].iloc[i]} with {top_cours["Students(thousands)"].iloc[i]} thousands of students enrolled.')

3. What are the most common words in course titles?

In [ ]:
titles_text = " ".join(df["Title"]).lower()
titles_text = re.sub(r"[^a-z\s]", "", titles_text)
stopwords = {
    "and", "or", "the", "to", "for", "of", "in", "with", "on", "a", "an",
    "by", "from", "at", "is", "are", "as", "into", "about", "your",
    "introduction", "learning", "science", "fundamentals", #words that don't really describe the nature of a course
    "de"
}

In [ ]:
words = [w for w in titles_text.split() if w not in stopwords]
word_counts = Counter(words)
most_common_words = word_counts.most_common(10)
most_common_words

We can see that not only technical topics such as data, Python, cloud, and development are highly popular on Coursera, but also non-technical fields like management, business, and health attract a large number of courses, highlighting the platform’s broad academic and professional focus.

**Certification Exploration**

4. What certification types are present in the dataset?

In [ ]:
res = ', '.join(df['Certificate'].unique())

In [ ]:
print(f"There are {len(df['Certificate'].unique())} certification types.\nHere is a list of all of them: {res.lower()}")

5. How many courses are offered per certification type?

In [ ]:
sns.countplot(data=df, x='Certificate', color=sns.color_palette()[0])
plt.title("Number of Courses per Certificate Type")
plt.xlabel("Certificate Type")
plt.ylabel("Number of Courses")

The most common certificate type on the platform is the standard Course, which largely dominates the dataset. It is followed by Specializations, while Professional Certificates represent only a small fraction of the total number of courses.

6. How many students are enrolled per certification type?

In [ ]:
grouped = df.groupby('Certificate')['Students(thousands)'].sum()

plt.figure(figsize=(8, 5))
bars = plt.bar(grouped.index, grouped.values)
for i, v in enumerate(grouped.values):
    plt.text(i, v - 2000, str(int(v)), ha='center', fontsize=10, color = "white")
    
plt.title("Number of total students enrolled(in thousands) per Certificate Type")
plt.xlabel("Certificate Type")
plt.ylabel("Students(thousands)")

plt.tight_layout()
plt.show()

The total number of enrolled students (in thousands) is highest for standard Courses, which attract more than 51 million learners overall. Specializations come next with around 27 million enrollments, while Professional Certificates have a significantly smaller student base. This suggests that standard course remain the most popular format on the platform, both in terms of availability and student demand.

7. What is the relationship between course difficulty and certification type?

In [ ]:
c_d = pd.crosstab(df['Certificate'], df['Difficulty'])

plt.figure(figsize=(8, 5))

sns.heatmap(c_d, annot=True, fmt="d", cmap="Blues")
plt.title("Relationship Between Course Difficulty and Certification Type")
plt.tight_layout()
plt.show()

Majority of courses across all certificate types are concentrated at the Beginner and Intermediate levels. Standard Courses dominate at every difficulty level, especially for beginners. Specializations are also mainly offered at beginner and intermediate levels, while Professional Certificates are relatively rare and almost entirely limited to beginner level.        
Overall, the number of courses decreases significantly as the difficulty increases

**Organization Exploration**

8. How many distinct organizations are in the dataset?

In [ ]:
print(f"There are {len(df['Organization'].unique())} organizators on coursera")

9. Which are the top 10 organizations by number of courses?

In [ ]:
max_cours = df['Organization'].value_counts()
max_cours = max_cours[:10]

print(f'Top organizations by number of courses :')
for i in range(10):
    print(f'{i+1}. {max_cours.index[i]} with {max_cours.iloc[i]} courses.')


In [ ]:
plt.figure(figsize=(8, 5))
bars = plt.barh(max_cours.index, max_cours.values)

    
plt.title("Top organizations by number of courses")
plt.xlabel("Number of courses")
plt.ylabel("Organizations")
plt.xlim(15, 60)
plt.tight_layout()

for bar in bars:
    width = bar.get_width()
    y_pos = bar.get_y() + bar.get_height() / 2
    plt.text(width - 3, y_pos, f"{int(width)}", va="center", color = "white")

plt.show()

10. Which organizations have the highest average course rating?

In [ ]:
max_rate = df.groupby('Organization')['Rating'].mean().sort_values(ascending = False)

max_rate

Many organizations have only one course that's why it's quite easy to maintain high score, this doesn't really get us any information. Let's look through our top 10 organizations by quantity and look at a relation there

11. What are the average course ratings for the top 10 organizations by number of courses?

In [ ]:
max_rate_top = df[df['Organization'].isin(max_cours.index)].groupby('Organization')['Rating'].mean().sort_values(ascending = False)
max_rate_top

In [ ]:
plt.figure(figsize=(8, 5))

bars = plt.barh(max_rate_top.index, max_rate_top.values)

    
plt.title("Average course ratings for the top 10 organizations")
plt.xlabel("Ratings")
plt.ylabel("Organizations")

plt.xlim(4.55, 4.75)

plt.tight_layout()

for bar in bars:
    width = bar.get_width()
    y_pos = bar.get_y() + bar.get_height() / 2
    plt.text(width - 0.02, y_pos, f"{width:.2f}", va="center", color="white")

plt.show()


12. How many students are enrolled in courses offered by the top 10 organizations?


In [ ]:
max_stud = df[df['Organization'].isin(max_cours.index)].groupby('Organization')['Students(thousands)'].sum().sort_values(ascending = False)

In [ ]:
plt.figure(figsize=(8, 5))

bars = plt.barh(max_stud.index, max_stud.values)

    
plt.title("Students enrolled in top 10 organizations")
plt.xlabel("Students(thousands)")
plt.ylabel("Organizations")

plt.tight_layout()

for bar in bars:
    width = bar.get_width()
    y_pos = bar.get_y() + bar.get_height() / 2
    plt.text(100, y_pos, f"{width:.0f}", va="center", color="white")

plt.show()

To compare these three metrics more effectively and improve the clarity of the visualization, let's use a slope chart to illustrate how the ranking of organizations changes across the different metrics.

13. How do the top 10 organizations compare across key metrics (number of courses, ratings, enrollments)?

In [ ]:
df_all = pd.DataFrame({
    "Students": max_stud,
    "Rating": max_rate_top,
    "Courses": max_cours
})

In [ ]:
##if vizualisation is needed
#df_all

In [ ]:
##sanity check to make sure dataframe is correct
# example = "University of Illinois at Urbana-Champaign"
# df[df['Organization'] == example]['Rating'].mean()
# df[df['Organization'] == example]['Students(thousands)'].sum()
# len(df[df['Organization'] == example])

In [ ]:
df_rank = df_all.rank(ascending=False, method="first")

In [ ]:
metrics = ["Courses", "Students", "Rating"]

plt.figure(figsize=(12,6))

for org in df_rank.index:
    plt.plot(metrics, df_rank.loc[org, metrics], marker="o", linewidth=2)

# Highest rank (1) at top
ax = plt.gca()
ax.invert_yaxis()
ax.get_yaxis().set_visible(False)

plt.title("Organization Comparison Across Metrics (Top 10 Organizations by nb of courses)")
plt.ylabel("Rank")
plt.xticks(metrics)
for org in df_rank.index:
    plt.text(
        x=-0.1,
        y=df_rank.loc[org, metrics][-1],
        s=org,
        fontsize=12,
        ha = 'right'
    )

plt.show()

We can see that there is no clear relationship between the metrics, especially between ratings and the number of students — the lines move a lot and do not follow a consistent pattern. What is clearly visible, however, is that there are two obvious leaders: the University of Michigan and the University of Pennsylvania. While the other organizations change their positions depending on the metric, these two consistently remain in the top 1–2 positions.

**Rating Exploration**

14. What is the distribution of course ratings?

In [ ]:
plt.figure(figsize=(12,4))
plt.subplot(1, 2, 1)
sns.histplot(data = df['Rating'])
plt.title("Distribution of Course Ratings")
plt.xlabel("Rating")
plt.ylabel("Count")

plt.subplot(1, 2, 2)
sns.boxplot(y=df['Rating'])
plt.title("Boxplot of Course Ratings")
plt.xlabel("Rating")
plt.show()

The distribution shows that most course ratings are concentrated between 4.5 and 4.8, indicating generally very high learner satisfaction. The boxplot confirms this by showing a tight interquartile range and only a few lower outliers, meaning that while a small number of courses receive lower ratings, the overall quality remains consistently high across the platform.

15. What is the relationship between course ratings and student enrollment?

We already observed with the top 10 organizations that there is no clear relationship between the number of students and course ratings. Let us now check whether this pattern also holds across the entire dataset using correlation

In [ ]:
df[['Rating', 'Students(thousands)']].corr()

We can now see that there is no relation between students enrolled and rating. Let's plot it.

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=df,
    x="Rating",
    y="Students(thousands)",
    alpha=0.6
)
plt.title("Rating vs Students Enrolled")
plt.show()

We can see that there is no strong relationship between course ratings and the number of enrolled students. Most courses have high ratings (between 4.5 and 5.0), but their enrollment levels vary widely, from very low to extremely high values. This indicates that a high rating does not necessarily guarantee a high number of students.

16. What is the relationship between course ratings and course difficulty?

In [ ]:
df.groupby('Difficulty')['Rating'].mean()

In [ ]:
plt.figure(figsize=(7,5))

df.boxplot(column='Rating', by='Difficulty',  medianprops=dict(color="red", linewidth=2))
plt.xlabel("Difficulty Level")
plt.ylabel("Rating")

plt.show()

We can see that course ratings are very similar across all difficulty levels, with comparable medians and spreads. This indicates that there is no clear relationship between course difficulty and rating

17. What is the relationship between course ratings and certification type?

In [ ]:
df.groupby('Certificate')['Rating'].mean()

In [ ]:
plt.figure(figsize=(7,5))

df.boxplot(column='Rating', by='Certificate',  medianprops=dict(color="red", linewidth=2))
plt.xlabel("Certificate")
plt.ylabel("Rating")

plt.show()

Similar results to course difficulties. This indicates that there is no clear relationship between course certificate and rating, however, what's interesting, we can see that professional certificates have no outliers

### Conclusion

Throughout the analysis, we examined the distribution of courses, certifications, organizations, student enrollments, and ratings on the platform. The dataset contains 891 courses offered by 154 different organizations. We found that popular courses span a wide range of fields, from technology to business and health, showing the diversity of learner interests.

The analysis also revealed that there is no strong relationship between most variables, particularly between course ratings, difficulty level, and the number of enrolled students. Highly rated courses are not necessarily the most popular, and more difficult courses are not rated differently from beginner ones. However, two organizations clearly stand out as overall leaders: the University of Pennsylvania and the University of Michigan. These institutions consistently dominate across all major indicators, having the largest number of courses, the highest ratings, and the most enrolled students.